# Cubos OLAP - Vistas Analíticas

Cinco vistas sobre el esquema estrella (`estrella`) que permiten análisis agregados.

| # | Vista | Dimensión principal |
|---|---|---|
| 1 | `cubo_ventas_por_tiempo` | Tiempo (año, mes) |
| 2 | `cubo_ventas_por_ubicacion` | Ubicación (restaurante) |
| 3 | `cubo_ventas_por_producto` | Tipo de producto (categoría) |
| 4 | `cubo_frecuencia_usuarios` | Frecuencia de uso (usuario) |
| 5 | `cubo_reservaciones_por_tiempo` | Tiempo + ocupación de mesas |

## 0. Configuración

In [1]:
HIVE_DB = "estrella"

## 1. Sesión de Spark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Esquema Estrella — MONGO")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sql(f"USE {HIVE_DB}")
print(f"Spark {spark.version} listo. Usando base '{HIVE_DB}'.")
spark.sql("SHOW TABLES IN estrella").show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 21:21:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/17 21:22:00 WARN HiveConf: HiveConf of name hive.metastore.local does not exist
26/06/17 21:22:00 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'spark' and will be reset to 'mr' to disable useless hive logic


Spark 4.1.2 listo. Usando base 'estrella'.
+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
| estrella|          dim_mesa|      false|
| estrella|         dim_plato|      false|
| estrella|   dim_restaurante|      false|
| estrella|        dim_tiempo|      false|
| estrella|       dim_usuario|      false|
| estrella|      fact_pedidos|      false|
| estrella|fact_reservaciones|      false|
+---------+------------------+-----------+



---
## Cubo 1 - Ventas por tiempo
Ingresos y cantidad de pedidos agrupados por año y mes.

In [3]:
spark.sql(f"DROP VIEW IF EXISTS {HIVE_DB}.cubo_ventas_por_tiempo")
spark.sql(f"""
    CREATE VIEW {HIVE_DB}.cubo_ventas_por_tiempo AS
    SELECT
        t.anio,
        t.mes,
        t.nombre_mes,
        COUNT(DISTINCT f.pedido_id)      AS total_pedidos,
        SUM(f.cantidad)                  AS total_items,
        ROUND(SUM(f.precio_total), 2)    AS ingresos_total,
        ROUND(AVG(f.precio_total), 2)    AS ticket_promedio
    FROM {HIVE_DB}.fact_pedidos f
    JOIN {HIVE_DB}.dim_tiempo t ON f.tiempo_sk = t.tiempo_sk
    WHERE f.estado != 'cancelado'
    GROUP BY t.anio, t.mes, t.nombre_mes
    ORDER BY t.anio, t.mes
""")
print("cubo_ventas_por_tiempo creado")
spark.sql(f"SELECT * FROM {HIVE_DB}.cubo_ventas_por_tiempo").show(truncate=False)

26/06/17 21:27:21 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


cubo_ventas_por_tiempo creado


[Stage 6:>                                                          (0 + 1) / 1]

+----+---+----------+-------------+-----------+--------------+---------------+
|anio|mes|nombre_mes|total_pedidos|total_items|ingresos_total|ticket_promedio|
+----+---+----------+-------------+-----------+--------------+---------------+
|2025|6  |June      |21           |105        |659827        |12449.57       |
|2025|7  |July      |42           |199        |1198126       |11746.33       |
|2025|8  |August    |40           |213        |1250769       |11581.19       |
|2025|9  |September |29           |146        |846463        |11922.01       |
|2025|10 |October   |30           |157        |989899        |13024.99       |
|2025|11 |November  |34           |169        |990208        |11788.19       |
|2025|12 |December  |38           |181        |1093494       |11158.1        |
|2026|1  |January   |49           |229        |1352438       |11365.03       |
|2026|2  |February  |25           |122        |702877        |10490.7        |
|2026|3  |March     |35           |135        |74997

---
## Cubo 2 - Ventas por ubicación
Ingresos y pedidos agrupados por restaurante y ubicación.

In [4]:
spark.sql("DROP VIEW IF EXISTS estrella.cubo_ventas_por_ubicacion")
spark.sql("""
    CREATE VIEW estrella.cubo_ventas_por_ubicacion AS
    SELECT
        r.nombre_rest                        AS restaurante,
        r.ubicacion,
        COUNT(DISTINCT f.pedido_id)          AS total_pedidos,
        SUM(f.cantidad)                      AS total_items,
        ROUND(SUM(f.precio_total), 2)        AS ingresos_total,
        ROUND(AVG(f.precio_total), 2)        AS ticket_promedio
    FROM estrella.fact_pedidos f
    JOIN estrella.fact_reservaciones rv ON f.usuario_sk = rv.usuario_sk
                                       AND f.tiempo_sk  = rv.tiempo_sk
    JOIN estrella.dim_mesa m            ON rv.mesa_sk = m.mesa_sk
    JOIN estrella.dim_restaurante r     ON m.restaurante = r.nombre_rest
    WHERE f.estado != 'cancelado'
    GROUP BY r.nombre_rest, r.ubicacion
    ORDER BY ingresos_total DESC
""")
print("cubo_ventas_por_ubicacion creado")
spark.sql("SELECT * FROM estrella.cubo_ventas_por_ubicacion LIMIT 10").show(truncate=False)

cubo_ventas_por_ubicacion creado


+--------------+---------+-------------+-----------+--------------+---------------+
|restaurante   |ubicacion|total_pedidos|total_items|ingresos_total|ticket_promedio|
+--------------+---------+-------------+-----------+--------------+---------------+
|El Viejo Roble|Cartago  |1            |8          |40967         |13655.67       |
|El Viejo Roble|Sarchí   |1            |8          |40967         |13655.67       |
|El Viejo Roble|Escazú   |1            |8          |40967         |13655.67       |
|Doña Sabroso  |Cartago  |1            |6          |36880         |12293.33       |
|Los los Sueños|San Pedro|1            |9          |36362         |9090.5         |
|Sabor Nuestro |Alajuela |2            |6          |35022         |8755.5         |
|La Fogata     |Escazú   |1            |7          |33457         |11152.33       |
|La Fogata     |Moravia  |1            |7          |33457         |11152.33       |
|La Fogata     |Tilaran  |1            |7          |33457         |11152.33 

---
## Cubo 3 - Ventas por tipo de producto
Ingresos y popularidad agrupados por categoría y plato.

In [5]:
spark.sql(f"DROP VIEW IF EXISTS {HIVE_DB}.cubo_ventas_por_producto")
spark.sql(f"""
    CREATE VIEW {HIVE_DB}.cubo_ventas_por_producto AS
    SELECT
        p.categoria,
        p.nombre_plato,
        p.precio,
        SUM(f.cantidad)                  AS total_vendido,
        ROUND(SUM(f.precio_total), 2)    AS ingresos_total,
        COUNT(DISTINCT f.pedido_id)      AS en_cuantos_pedidos
    FROM {HIVE_DB}.fact_pedidos f
    JOIN {HIVE_DB}.dim_plato p ON f.plato_sk = p.plato_sk
    WHERE f.estado != 'cancelado'
    GROUP BY p.categoria, p.nombre_plato, p.precio
    ORDER BY total_vendido DESC
""")
print("cubo_ventas_por_producto creado")
spark.sql(f"SELECT * FROM {HIVE_DB}.cubo_ventas_por_producto LIMIT 10").show(truncate=False)

cubo_ventas_por_producto creado


[Stage 19:>                                                         (0 + 1) / 1]

+---------+-----------------------------+------+-------------+--------------+------------------+
|categoria|nombre_plato                 |precio|total_vendido|ingresos_total|en_cuantos_pedidos|
+---------+-----------------------------+------+-------------+--------------+------------------+
|Cenas    |Camarones al ajillo          |9966  |9            |89694         |3                 |
|Cenas    |Cordero con hierbas          |13448 |9            |121032        |3                 |
|Bebidas  |Refresco natural             |1551  |8            |12408         |3                 |
|Sopas    |Sopa de tomate               |3603  |8            |28824         |3                 |
|Bebidas  |Batido de mango              |2236  |7            |15652         |3                 |
|Bebidas  |Refresco natural             |1443  |6            |8658          |2                 |
|Mariscos |Cangrejo en salsa de coco    |12462 |6            |74772         |3                 |
|Almuerzos|Chifrijo           

---
## Cubo 4 - Frecuencia de uso por usuario
Cuántas veces cada usuario ha pedido, cuánto ha gastado y cuándo fue su última orden.

In [6]:
spark.sql(f"DROP VIEW IF EXISTS {HIVE_DB}.cubo_frecuencia_usuarios")
spark.sql(f"""
    CREATE VIEW {HIVE_DB}.cubo_frecuencia_usuarios AS
    SELECT
        u.nombre,
        u.email,
        COUNT(DISTINCT f.pedido_id)       AS total_pedidos,
        SUM(f.cantidad)                   AS total_items,
        ROUND(SUM(f.precio_total), 2)     AS gasto_total,
        ROUND(AVG(f.precio_total), 2)     AS gasto_promedio_por_item,
        MAX(t.fecha)                      AS ultima_orden
    FROM {HIVE_DB}.fact_pedidos f
    JOIN {HIVE_DB}.dim_usuario u  ON f.usuario_sk = u.usuario_sk
    JOIN {HIVE_DB}.dim_tiempo  t  ON f.tiempo_sk  = t.tiempo_sk
    WHERE f.estado != 'cancelado'
    GROUP BY u.nombre, u.email
    ORDER BY gasto_total DESC
""")
print("cubo_frecuencia_usuarios creado")
spark.sql(f"SELECT * FROM {HIVE_DB}.cubo_frecuencia_usuarios LIMIT 10").show(truncate=False)

cubo_frecuencia_usuarios creado


[Stage 25:==========================================================(2 + 0) / 2]

+------+-----+-------------+-----------+-----------+-----------------------+------------+
|nombre|email|total_pedidos|total_items|gasto_total|gasto_promedio_por_item|ultima_orden|
+------+-----+-------------+-----------+-----------+-----------------------+------------+
+------+-----+-------------+-----------+-----------+-----------------------+------------+



---
## Cubo 5 - Reservaciones por tiempo y ocupación
Distribución de reservaciones por mes y hora, con tasa de ocupación de mesas.

In [7]:
spark.sql(f"DROP VIEW IF EXISTS {HIVE_DB}.cubo_reservaciones_por_tiempo")
spark.sql(f"""
    CREATE VIEW {HIVE_DB}.cubo_reservaciones_por_tiempo AS
    SELECT
        t.anio,
        t.mes,
        t.nombre_mes,
        CAST(SUBSTRING(r.hora_reservacion, 1, 2) AS INT)  AS hora,
        m.restaurante,
        COUNT(*)                                           AS total_reservaciones,
        SUM(CASE WHEN r.estado = 'completada' THEN 1 ELSE 0 END) AS completadas,
        SUM(CASE WHEN r.estado = 'cancelado'  THEN 1 ELSE 0 END) AS canceladas,
        ROUND(
            SUM(CASE WHEN r.estado = 'completada' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        ) AS tasa_completadas_pct
    FROM {HIVE_DB}.fact_reservaciones r
    JOIN {HIVE_DB}.dim_tiempo t ON r.tiempo_sk = t.tiempo_sk
    JOIN {HIVE_DB}.dim_mesa   m ON r.mesa_sk   = m.mesa_sk
    GROUP BY t.anio, t.mes, t.nombre_mes,
             CAST(SUBSTRING(r.hora_reservacion, 1, 2) AS INT),
             m.restaurante
    ORDER BY t.anio, t.mes, hora
""")
print("cubo_reservaciones_por_tiempo creado")
spark.sql(f"SELECT * FROM {HIVE_DB}.cubo_reservaciones_por_tiempo LIMIT 10").show(truncate=False)

cubo_reservaciones_por_tiempo creado


[Stage 30:=============================>                            (1 + 1) / 2]

+----+---+----------+----+--------------+-------------------+-----------+----------+--------------------+
|anio|mes|nombre_mes|hora|restaurante   |total_reservaciones|completadas|canceladas|tasa_completadas_pct|
+----+---+----------+----+--------------+-------------------+-----------+----------+--------------------+
|2025|6  |June      |11  |Mar y Tropical|1                  |0          |0         |0.00                |
|2025|6  |June      |12  |Donde Marta   |1                  |1          |0         |100.00              |
|2025|6  |June      |14  |La Fogata     |1                  |0          |0         |0.00                |
|2025|6  |June      |15  |Tierra Viva   |1                  |1          |0         |100.00              |
|2025|6  |June      |16  |El Bohío      |1                  |1          |0         |100.00              |
|2025|6  |June      |18  |Doña Auténtico|1                  |0          |0         |0.00                |
|2025|6  |June      |18  |La Tinaja     |1    

---
## Resumen de vistas

In [8]:
print(f"Vistas en la base de datos '{HIVE_DB}':")
spark.sql(f"SHOW VIEWS IN {HIVE_DB}").show(truncate=False)

Vistas en la base de datos 'estrella':
+---------+-----------------------------+-----------+
|namespace|viewName                     |isTemporary|
+---------+-----------------------------+-----------+
|estrella |cubo_frecuencia_usuarios     |false      |
|estrella |cubo_reservaciones_por_tiempo|false      |
|estrella |cubo_ventas_por_producto     |false      |
|estrella |cubo_ventas_por_tiempo       |false      |
|estrella |cubo_ventas_por_ubicacion    |false      |
+---------+-----------------------------+-----------+



In [9]:
spark.stop()
print("Sesión cerrada.")

Sesión cerrada.


26/06/17 21:29:25 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-f89c9109-606f-44d3-9ab0-f9d73c68921d. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-f89c9109-606f-44d3-9ab0-f9d73c68921d
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:67)
	at scala.collection.ArrayOps$.foreach$exten